In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/pambifree/05-finetune/__huggingface_repos__.json
/kaggle/input/notebooks/pambifree/05-finetune/__notebook__.ipynb
/kaggle/input/notebooks/pambifree/05-finetune/__output__.json
/kaggle/input/notebooks/pambifree/05-finetune/__results__.html
/kaggle/input/notebooks/pambifree/05-finetune/custom.css
/kaggle/input/notebooks/pambifree/05-finetune/finetune_run_log.csv
/kaggle/input/notebooks/pambifree/05-finetune/finetune_runs.csv
/kaggle/input/notebooks/pambifree/05-finetune/probs/HingBERT_seed42.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/HingBERT_seed43.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/HingBERT_seed44.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/XLM-R_base_seed42.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/XLM-R_base_seed43.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/XLM-R_base_seed44.npz
/kaggle/input/notebooks/pambifree/02-data-audit/__huggingface_repos__.json
/kaggle/input/notebooks/pambifree/02

In [2]:
!find /kaggle/input -name "*.npz"

/kaggle/input/notebooks/pambifree/05-finetune/probs/HingBERT_seed42.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/HingBERT_seed43.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/HingBERT_seed44.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/XLM-R_base_seed42.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/XLM-R_base_seed43.npz
/kaggle/input/notebooks/pambifree/05-finetune/probs/XLM-R_base_seed44.npz


In [3]:
!find /kaggle/input -name "*.csv" | head -20

/kaggle/input/notebooks/pambifree/05-finetune/finetune_run_log.csv
/kaggle/input/notebooks/pambifree/05-finetune/finetune_runs.csv
/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/test.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/train.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/val.csv


In [4]:
import numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

PUBLIC_DIR = "/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1"     # replace
GOLD_PATH = "/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv"    # replace
PROBS_DIR = "/kaggle/input/notebooks/pambifree/05-finetune/probs"               # replace

train = pd.read_csv(f"{PUBLIC_DIR}/train.csv"); val = pd.read_csv(f"{PUBLIC_DIR}/val.csv")
test = pd.read_csv(f"{PUBLIC_DIR}/test.csv"); gold = pd.read_csv(GOLD_PATH)

SEED = 42
def cap_groups(df, cap=5, seed=SEED):
    return df.sample(frac=1, random_state=seed).groupby("group").head(cap)
val_c, test_c = cap_groups(val), cap_groups(test)

def scores(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    legit_ok = ((y_pred == 0) & (y_true == 0)).sum() / max((y_true == 0).sum(), 1)
    return {"precision": p, "recall": r, "f1": f, "accuracy": accuracy_score(y_true, y_pred),
            "false_alarm_rate": 1 - legit_ok, "balanced_acc": (r + legit_ok) / 2}

def load_probs(name, seeds=(42, 43, 44)):
    arrs = [np.load(f"{PROBS_DIR}/{name}_seed{s}.npz") for s in seeds]
    return {k: np.mean([a[k] for a in arrs], axis=0) for k in ["val", "test", "gold"]}

probs = {"HingBERT FT (3-seed avg)": load_probs("HingBERT"),
         "XLM-R FT (3-seed avg)": load_probs("XLM-R_base")}

# baseline trained on train only, same as the fine-tuned models
base = Pipeline([("vec", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
                 ("clf", LogisticRegression(max_iter=2000))]).fit(train["text"], train["is_scam"])
probs["TF-IDF chars + LogReg"] = {"val": base.predict_proba(val_c["text"])[:, 1],
                                  "test": base.predict_proba(test["text"])[:, 1],
                                  "gold": base.predict_proba(gold["text"])[:, 1]}
print({k: {s: len(v) for s, v in d.items()} for k, d in probs.items()})

{'HingBERT FT (3-seed avg)': {'val': 1350, 'test': 2066, 'gold': 326}, 'XLM-R FT (3-seed avg)': {'val': 1350, 'test': 2066, 'gold': 326}, 'TF-IDF chars + LogReg': {'val': 1350, 'test': 2066, 'gold': 326}}


**Step 2. Test the hypothesis on the training data**

In [5]:
link_re = r"https?://|www\.|\b[\w-]+\.(?:com|in|net|org|co|info|xyz|live|me)\b"
alert_re = r"\bOTP\b|PNR|credited|debited|recharge|a/c|balance|alert|invoice|order|delivered"
tr = train.assign(has_link=train["text"].str.contains(link_re, case=False, regex=True),
                  alert_words=train["text"].str.contains(alert_re, case=False, regex=True))

print("Share of training messages containing a link (0 = legit, 1 = unwanted):")
print(tr.groupby("is_scam")["has_link"].mean().round(3), "\n")
print("Share with alert-style words:")
print(tr.groupby("is_scam")["alert_words"].mean().round(3), "\n")

legit = tr[tr["is_scam"] == 0]
print("Legit training rows with a link:", int(legit["has_link"].sum()), "of", len(legit))
print(legit.groupby("source_dataset")[["has_link", "alert_words"]].mean().round(3).assign(rows=legit["source_dataset"].value_counts()))

Share of training messages containing a link (0 = legit, 1 = unwanted):
is_scam
0    0.014
1    0.243
Name: has_link, dtype: float64 

Share with alert-style words:
is_scam
0    0.216
1    0.181
Name: alert_words, dtype: float64 

Legit training rows with a link: 52 of 3600
                                    has_link  alert_words  rows
source_dataset                                                 
Indian_Telecom_SMS                     0.000        0.000    79
Smishing_Dataset                       0.021        0.014  2499
Synthetic_Tier_C                       0.000        1.000   741
UCI_SMS_Spam                           0.000        0.000   270
ysangam/Indian_Cyber_Scam_Hinglish     0.000        0.273    11


**Step 3. Confidence intervals on gold**

In [6]:
y_gold = gold["label"].values
rng = np.random.default_rng(0)
idx = rng.integers(0, len(y_gold), size=(2000, len(y_gold)))

def rates(y, pred):
    pos, neg = y == 1, y == 0
    r, fa = (pred[pos] == 1).mean(), (pred[neg] == 1).mean()
    return r, fa, (r + 1 - fa) / 2

def boot(y, pred):
    return np.array([rates(y[b], pred[b]) for b in idx])

gpred = {k: (v["gold"] >= 0.5).astype(int) for k, v in probs.items()}
boots = {k: boot(y_gold, p) for k, p in gpred.items()}

rows = []
for k, B in boots.items():
    r, fa, ba = rates(y_gold, gpred[k])
    lo, hi = np.percentile(B, [2.5, 97.5], axis=0)
    rows.append({"model": k, "recall": f"{r:.3f} [{lo[0]:.2f}-{hi[0]:.2f}]",
                 "false_alarm": f"{fa:.3f} [{lo[1]:.2f}-{hi[1]:.2f}]",
                 "balanced_acc": f"{ba:.3f} [{lo[2]:.2f}-{hi[2]:.2f}]"})
display(pd.DataFrame(rows).set_index("model"))

base_name = "TF-IDF chars + LogReg"
for k in ["HingBERT FT (3-seed avg)", "XLM-R FT (3-seed avg)"]:
    d = boots[k][:, 2] - boots[base_name][:, 2]
    print(f"{k} minus baseline, balanced accuracy on gold: {d.mean():+.3f}, 95% CI {np.percentile(d, [2.5, 97.5]).round(3)}")


,recall,false_alarm,balanced_acc
model,,,
HingBERT FT (3-seed avg),0.869 [0.82-0.91],0.524 [0.45-0.61],0.672 [0.62-0.72]
XLM-R FT (3-seed avg),0.836 [0.78-0.89],0.524 [0.44-0.61],0.656 [0.61-0.70]
TF-IDF chars + LogReg,0.836 [0.78-0.89],0.524 [0.44-0.61],0.656 [0.61-0.71]


HingBERT FT (3-seed avg) minus baseline, balanced accuracy on gold: +0.016, 95% CI [-0.024  0.054]
XLM-R FT (3-seed avg) minus baseline, balanced accuracy on gold: -0.001, 95% CI [-0.045  0.044]


**Step 4. Does a stricter threshold help?**

In [7]:
name = "HingBERT FT (3-seed avg)"
yv = val_c["is_scam"].values
thr_val = float(np.quantile(probs[name]["val"][yv == 0], 0.95))
print(f"Threshold giving ~5% false alarms on validation: {thr_val:.3f}\n")

sets = [("val (capped)", probs[name]["val"], yv),
        ("test (capped)", probs[name]["test"][test_c.index], test_c["is_scam"].values),
        ("gold", probs[name]["gold"], y_gold)]
for set_name, p, y in sets:
    for t in (0.5, thr_val):
        s = scores(y, (p >= t).astype(int))
        print(f"{set_name:14} threshold {t:.3f}: recall {s['recall']:.3f}, false alarms {s['false_alarm_rate']:.3f}, balanced acc {s['balanced_acc']:.3f}")

print("\nGold sweep (information only, not for choosing a threshold):")
for t in (0.5, 0.7, 0.9, 0.95, 0.99):
    s = scores(y_gold, (probs[name]["gold"] >= t).astype(int))
    print(f"  threshold {t:.2f}: recall {s['recall']:.3f}, false alarms {s['false_alarm_rate']:.3f}, balanced acc {s['balanced_acc']:.3f}")

Threshold giving ~5% false alarms on validation: 0.072

val (capped)   threshold 0.500: recall 0.981, false alarms 0.030, balanced acc 0.975
val (capped)   threshold 0.072: recall 0.990, false alarms 0.051, balanced acc 0.970
test (capped)  threshold 0.500: recall 0.982, false alarms 0.057, balanced acc 0.963
test (capped)  threshold 0.072: recall 0.996, false alarms 0.075, balanced acc 0.960
gold           threshold 0.500: recall 0.869, false alarms 0.524, balanced acc 0.672
gold           threshold 0.072: recall 0.940, false alarms 0.594, balanced acc 0.673

Gold sweep (information only, not for choosing a threshold):
  threshold 0.50: recall 0.869, false alarms 0.524, balanced acc 0.672
  threshold 0.70: recall 0.825, false alarms 0.483, balanced acc 0.671
  threshold 0.90: recall 0.770, false alarms 0.413, balanced acc 0.679
  threshold 0.95: recall 0.710, false alarms 0.371, balanced acc 0.670
  threshold 0.99: recall 0.607, false alarms 0.238, balanced acc 0.684


**Save and commit**

In [8]:
pd.DataFrame(rows).to_csv("/kaggle/working/gold_confidence_intervals.csv", index=False)